In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['GEMINI_API_KEY']:
    print("GEMINI_API_KEY is set.")

GEMINI_API_KEY is set.


In [2]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
llm= ChatGoogleGenerativeAI(model="gemini-2.5-flash",
                            api_key=os.environ['GEMINI_API_KEY'])

In [11]:
llm= ChatOpenAI(model="gemini-2.5-flash",
                api_key=os.environ['GEMINI_API_KEY'])

In [18]:
response=llm.invoke("What is the capital of France?")
response.content

'The capital of France is **Paris**.'

## RAG IMPLEMENTATION WITH OWN TEXT DATA

# 
Step -1 Preparing document for text


In [ ]:
from langchain_core.documents import Document

my_text = """ Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers have used techniques including state space search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind and Meta, aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human.[3]

Artificial intelligence was founded as an academic discipline in 1956,[4] and the field went through multiple cycles of optimism throughout its history,[5][6] followed by periods of disappointment and loss of funding, known as AI winters.[7][8] Funding and interest increased substantially after 2012, when graphics processing units began being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[9] This growth accelerated further after 2017 with the transformer architecture.[10] In the 2020s, an AI boom has coincided with advances in generative AI, which allowed for the creation and modification of media. In addition to AI safety and unintended consequences and harms from the use of AI, ethical concerns, AI's long-term effects, and potential existential risks have prompted discussions of AI regulation."""

doc = [Document(page_content=my_text, metadata={"source": "Wikipedia"})]
doc


[Document(metadata={'source': 'Wikipedia'}, page_content=" Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]\n\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.\n\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and percep

# 
Step-2 Chunking

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [25]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks=splitter.split_documents(doc)
chunks

[Document(metadata={'source': 'Wikipedia'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'Wikipedia'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.'),
 Document(metadata={'source': 'Wikipedia'}, page_content='The traditional goals of A

#
Step-3 Embedding of chunks


In [26]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
      
)

#
Step-4 Create and Store the Embeddings in Vector DB

In [45]:
from langchain_community.vectorstores import Chroma


vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)


In [46]:
vectors=[]
for doc in chunks:
    vector=embeddings.embed_query(doc.page_content)
    vectors.append(vector)
vectors

[[-0.010689758,
  0.004775691,
  0.0038520142,
  0.022690848,
  -0.0011877314,
  0.006196375,
  -0.009706793,
  -0.001972661,
  -0.009426111,
  -0.044186514,
  -0.010057568,
  0.015600424,
  -0.00630443,
  0.0049365847,
  0.009117768,
  -0.0102861645,
  0.0206042,
  -0.011045287,
  0.014706382,
  0.0028224315,
  0.00343372,
  0.02738449,
  0.032144427,
  0.010373725,
  -0.022851003,
  0.021842642,
  0.0024993601,
  -0.014371862,
  -0.01662891,
  0.12013936,
  -0.008669156,
  -0.01769768,
  0.017740065,
  -0.016626485,
  0.0160869,
  0.0125664845,
  -0.010009229,
  -0.024398642,
  0.012706488,
  -0.017698992,
  -0.00505511,
  0.010053089,
  0.011923229,
  0.010378752,
  0.0032468191,
  -0.012996217,
  -0.0515598,
  -0.011068271,
  -0.0011467558,
  -0.007980938,
  -0.008716677,
  0.0044720625,
  0.017670497,
  -0.031739928,
  0.020654168,
  0.010521535,
  0.02860541,
  0.0019669675,
  -0.00459204,
  -0.011512861,
  -0.00542573,
  0.024789624,
  0.014156036,
  0.021049997,
  0.028785752,


#
Step-5 Semantic Search

In [47]:
vectorstore.similarity_search("What is AI?", k=3)

[Document(metadata={'source': 'Wikipedia'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'Wikipedia'}, page_content='The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers have used techniques including state space search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operation